# 01 — TF-IDF a parità di genere (Versione 3)

Lo stesso TF-IDF + regressione logistica della Versione 2, ma con il compito **binario** (affidabile = punteggio
NewsGuard ≥ 60) e con i **pesi a parità di genere**: dentro ogni orientamento politico NewsGuard (Far Right, Slightly
Right, Slightly Left, Far Left, nessuno) le testate affidabili e quelle inaffidabili pesano uguale nel training.
Così l'orientamento, il "genere" della testata, non aiuta a indovinare.

- Training: tutte le testate dei gruppi di training (anche quelle di solo training della Versione 2), con il peso del
  gruppo (al più 5 testate) per il fattore di genere.
- Test: solo le 304 testate valutate, fold per gruppo (gli stessi della Versione 2).
- Testata = media delle probabilità di "affidabile" dei suoi articoli; soglia 0,5.
- Metrica principale (fissata prima): **media sugli orientamenti** dell'accuratezza bilanciata. I soli metadati
  (l'orientamento) fanno 0,50 per costruzione.
- Con `C_GRIGLIA` il C si sceglie in una validazione interna per gruppi, sul solo training, con la stessa metrica.

Si salva tutto: per articolo `article_id, domain, fold, y, orientamento, logit, p`; `metriche.json`; `FATTO`.

In [1]:
import json
import time

import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedGroupKFold

from comune import RESULTS, SEED, carica, fattori_genere, metriche, per_testata, pesi_righe, registratore

In [2]:
NOME = "feature_genere"
C_GRIGLIA = ""             # "" = C = 1; "0.1,1,10,100" = C scelto in validazione interna (solo training)
N_JOBS = 5
PROVA = False

In [3]:
# Parameters
NOME = "feature_genere"


In [4]:
OUT = (RESULTS / "prova" if PROVA else RESULTS) / NOME
OUT.mkdir(parents=True, exist_ok=True)
registra = registratore(OUT.parent / f"avanzamento_{NOME}.log")
art = carica(PROVA)
registra(f"=== {NOME}: {len(art)} articoli, {art.domain.nunique()} testate ({art[art.valutazione].domain.nunique()} valutate), C {C_GRIGLIA or 1} ===")
f0 = fattori_genere(art)
registra("fattori di genere (su tutto, solo per il log): " + ", ".join(f"{o}/{'aff' if c else 'inaff'} {v:.2f}" for (o, c), v in sorted(f0.items())))


def tfidf():
    return TfidfVectorizer(ngram_range=(1, 2), min_df=5 if not PROVA else 2, max_df=0.9, sublinear_tf=True,
                           max_features=100_000, strip_accents="unicode")


def adatta(a, c):
    v = tfidf(); X = v.fit_transform(a.text)
    w = pesi_righe(a, fattori_genere(a))
    return v, LogisticRegression(C=c, max_iter=2000, random_state=SEED).fit(X, a.y, sample_weight=w)


def scegli_C(tr):
    griglia = [float(x) for x in C_GRIGLIA.split(",")]
    t = tr.drop_duplicates("domain").reset_index(drop=True)
    punti = {c: [] for c in griglia}
    for itr, iva in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(t, t.orientamento + t.y.astype(str), t.gruppo):
        a = tr[tr.domain.isin(t.domain[itr])]
        b = tr[tr.domain.isin(t.domain[iva]) & tr.valutazione]
        for c in griglia:
            v, clf = adatta(a, c)
            b2 = b.assign(p=clf.predict_proba(v.transform(b.text))[:, 1])
            punti[c].append(metriche(per_testata(b2))["media sugli orientamenti"])
    medie = {c: float(np.nanmean(s)) for c, s in punti.items()}
    return max(medie, key=medie.get), medie


def un_fold(f):
    t0 = time.time()
    tr, te = art[art.fold_gruppo != f], art[(art.fold_gruppo == f) & art.valutazione]
    C, medie = (1.0, {}) if not C_GRIGLIA else scegli_C(tr)
    v, clf = adatta(tr, C)
    X = v.transform(te.text)
    out = te[["article_id", "domain", "fold_gruppo", "y", "orientamento"]].rename(columns={"fold_gruppo": "fold"}).copy()
    out["logit"] = clf.decision_function(X); out["p"] = clf.predict_proba(X)[:, 1]
    m = metriche(per_testata(out))
    return f, out, C, medie, m, time.time() - t0


parti = []
for f, out, C, medie, m, sec in Parallel(n_jobs=N_JOBS)(delayed(un_fold)(f) for f in sorted(art.fold_gruppo.unique())):
    parti.append(out)
    registra(f"fold {f}: C {C}" + (f" (interne {', '.join(f'{k}: {x:.3f}' for k, x in medie.items())})" if medie else "")
             + f"; media sugli orientamenti {m['media sugli orientamenti']:.3f}, AUC {m['AUC media sugli orientamenti']:.3f} ({sec / 60:.0f} min)")
pred = pd.concat(parti).sort_values("article_id")
pred.to_csv(OUT / "articoli.csv", index=False)
m = metriche(per_testata(pred))
json.dump(m, open(OUT / "metriche.json", "w"), indent=1)
(OUT / "FATTO").write_text(time.strftime("%Y-%m-%d %H:%M:%S"))
registra(f"=== {NOME} FINITO: media sugli orientamenti {m['media sugli orientamenti']:.3f}, AUC {m['AUC media sugli orientamenti']:.3f}, "
         f"bilanciata su tutto {m['bilanciata su tutto']:.3f} ===")
pd.Series(m).round(3)

2026-10-20 12:24:09  === feature_genere: 75389 articoli, 863 testate (304 valutate), C 1 ===


2026-10-20 12:24:09  fattori di genere (su tutto, solo per il log): (nessuno)/inaff 2.86, (nessuno)/aff 0.61, Far Left/inaff 1.75, Far Left/aff 0.70, Far Right/inaff 0.57, Far Right/aff 3.90, Slightly Left/inaff 3.61, Slightly Left/aff 0.58, Slightly Right/inaff 1.03, Slightly Right/aff 0.97


2026-10-20 12:27:24  fold 1: C 1.0; media sugli orientamenti 0.653, AUC 0.687 (3 min)


2026-10-20 12:27:24  fold 2: C 1.0; media sugli orientamenti 0.615, AUC 0.764 (3 min)


2026-10-20 12:27:24  fold 3: C 1.0; media sugli orientamenti 0.564, AUC 0.725 (3 min)


2026-10-20 12:27:24  fold 4: C 1.0; media sugli orientamenti 0.705, AUC 0.735 (3 min)


2026-10-20 12:27:24  fold 5: C 1.0; media sugli orientamenti 0.580, AUC 0.642 (3 min)


2026-10-20 12:27:25  === feature_genere FINITO: media sugli orientamenti 0.618, AUC 0.671, bilanciata su tutto 0.703 ===


testate                         304.000
media sugli orientamenti          0.618
AUC media sugli orientamenti      0.671
bilanciata (nessuno)              0.707
bilanciata Far Left               0.625
bilanciata Far Right              0.650
bilanciata Slightly Left          0.544
bilanciata Slightly Right         0.562
bilanciata su tutto               0.703
accuratezza                       0.714
F1 macro                          0.704
dtype: float64